# S1GHT native runtime on GPU

GPU checks and measurements for S1GHT's native runtime, on Colab or Kaggle. Each section says which branch it needs:

1. `pytest -m gpu`: for d1-3B and d1-omni-600M, the catalog dtype (bf16 on A100-class GPUs, fp16 on a T4) must give the same answers as fp32 on the same GPU.
2. A latency table: one question, three questions, and a 384 px image, warm, median of 20 runs.
3. Qwen3-VL-4B: native vs vLLM.
4. Latency by image size and batch (`s1ght bench`).
5. Qwen3-VL-4B prefix cache.
6. P2.0: location signal in the patch hidden states.
7. P2.1: a trained detection head on the frozen model, COCO mAP.

**Setup**
- Colab: *Runtime → Change runtime type → GPU*. Kaggle: *Settings → Accelerator → GPU*.
- Set `BRANCH` below to the branch the section needs (`main` for released code), then run the GPU check, clone and install cells.
- The repository is public, so no token is needed. The released package is on PyPI as `s1ght-visual-decisions`, but these sections run branch code, tests and `benchmarks/` scripts, so the notebook installs from a clone of `BRANCH`.

Models download from Hugging Face at the commits pinned in `registry.yaml`; each section lists its downloads.

In [ ]:
BRANCH = "feat/patch-head"  # section 7; 6 ran on "feat/patch-signal", 5 on "feat/prefix-cache", 4 on "feat/bench", 1-2 on "p1/native-d1"
MODELS = ["d1-omni-600m", "d1-3b"]

In [ ]:
import subprocess

import torch

query = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"]
print(subprocess.run(query, check=False, capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "no GPU: change the runtime type"
print(torch.__version__, torch.cuda.get_device_name(), "compute capability", torch.cuda.get_device_capability())

In [ ]:
import shutil
import subprocess

# public repository: no token needed; the branch's tests and benchmarks/ are not in the PyPI package
shutil.rmtree("S1GHT", ignore_errors=True)
clone = ["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, "https://github.com/Doga0/S1GHT.git", "S1GHT"]
r = subprocess.run(clone, check=False, capture_output=True, text=True)
print("cloned", BRANCH if r.returncode == 0 else r.stderr)

In [ ]:
# A regular (non-editable) install: the running kernel sees it at once. An editable install
# adds a .pth file that Python only reads at start-up, so `import s1ght` would need a restart.
%pip install -q "./S1GHT[native,dev]"
import importlib

importlib.invalidate_caches()
import s1ght

print("s1ght", s1ght.__version__, "from", s1ght.__file__)

## 1. Catalog dtype vs fp32

In [ ]:
!cd S1GHT && python -m pytest tests/test_runtime_native.py -m gpu -s -q -rs -p no:cacheprovider

## 2. Latency (warm, median of 20)

In [ ]:
import gc
import statistics
import time

import torch
from PIL import Image as PILImage
from PIL import ImageDraw

from s1ght import S1GHT, Choice, Image, Noul, Score


def drawing(size=384):
    canvas = PILImage.new("RGB", (size, size), "white")
    ImageDraw.Draw(canvas).ellipse((size // 4, size // 4, 3 * size // 4, 3 * size // 4), fill=(220, 20, 20))
    return canvas


TEXT = "I was charged twice this month, please refund one of them."
ONE = {"refund": Noul("Is the customer asking for a refund?")}
THREE = {
    **ONE,
    "team": Choice(
        "Which team should handle this?",
        {"billing": "Charges, refunds", "technical": "Faults", "fraud": "Unauthorised use"},
    ),
    "urgency": Score("How urgent is this?", ["Can wait", "Today", "Blocking now"]),
}
IMAGE_Q = {"color": Choice("What color is the main shape?", {"red": None, "green": None, "blue": None})}


def median_ms(fn, n=20, warmup=3):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(n):
        torch.cuda.synchronize()
        t = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)


rows = []
for name in MODELS:
    torch.cuda.reset_peak_memory_stats()
    with S1GHT(name, device="cuda") as m:
        img = Image(drawing())
        rows.append(
            (
                name,
                m.runtime.dtype,
                median_ms(lambda m=m: m.decide(TEXT, ONE)),
                median_ms(lambda m=m: m.decide(TEXT, THREE)),
                median_ms(lambda m=m, img=img: m.decide({"photo": img}, IMAGE_Q)),
                torch.cuda.max_memory_allocated() / 2**30,
            )
        )
    gc.collect()
    torch.cuda.empty_cache()

gpu = torch.cuda.get_device_name()
print(gpu)
print(f"{'model':14s} {'dtype':5s} {'1 q':>8s} {'3 q':>8s} {'384px img':>10s} {'peak VRAM':>10s}")
for name, dtype, a, b, c, mem in rows:
    print(f"{name:14s} {dtype:5s} {a:6.1f}ms {b:6.1f}ms {c:8.1f}ms {mem:8.2f}GB")

Paste the output of the two sections above into the PR or an issue. The models stay in this machine's Hugging Face cache until the runtime is recycled.

## 3. Qwen3-VL-4B: native vs vLLM (P1 step 11)

Run this section in a **fresh runtime**: set `BRANCH = "p1/qwen-vllm"` in the first cell, run the GPU check, clone and install cells, then these. Installing vLLM replaces the runtime's torch.

Both paths send the same prompt and the same pixels (S1GHT resizes images before either sees them), so the answers should match up to fp16 noise. vLLM runs first and is stopped before the native model loads, because one T4 (15 GB) cannot hold both. Downloads: Qwen3-VL-4B-Instruct 8.9 GB (once, shared by both).

Start vLLM (pinned to the catalog's commit; fp16 on T4, bf16 on A100) and wait until it answers.

In [ ]:
%pip install -q vllm
import subprocess
import time  # noqa: F811 - section 3 runs in a fresh runtime, without section 2's imports
from pathlib import Path

import httpx

from s1ght.hub import resolve

REPO, REV = "Qwen/Qwen3-VL-4B-Instruct", resolve("qwen3-vl-4b").source.revision
gpu_bf16 = torch.cuda.get_device_capability()[0] >= 8
vllm_log = open("vllm.log", "w")  # noqa: SIM115 - stays open for the server's lifetime
server = subprocess.Popen(
    ["vllm", "serve", REPO, "--revision", REV, "--dtype", "bfloat16" if gpu_bf16 else "half",
     "--max-model-len", "8192", "--gpu-memory-utilization", "0.85", "--max-logprobs", "20", "--port", "8000"],
    stdout=vllm_log, stderr=subprocess.STDOUT,
)  # fmt: skip
t0 = time.time()
while True:
    try:
        if httpx.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
            break
    except httpx.TransportError:
        pass
    if server.poll() is not None or time.time() - t0 > 1200:
        raise RuntimeError("vLLM did not start:\n" + Path("vllm.log").read_text()[-3000:])
    time.sleep(5)
print(f"vLLM ready in {time.time() - t0:.0f}s")

In [ ]:
import gc

import torch

from s1ght import S1GHT, Choice, Image, Noul, Score

PHOTOS = {n: Image(httpx.get(f"https://ultralytics.com/images/{n}", follow_redirects=True).content)
          for n in ("bus.jpg", "zidane.jpg")}  # fmt: skip
QS = {
    "bus": Noul("Is there a bus in `photo`?"),
    "scene": Choice("Where was `photo` taken?", ["city street", "football stadium", "kitchen", "beach"]),
    "people": Score("How many people are in `photo`?", ["none", "one or two", "three or more"]),
}


def summary(r):
    a = r.answers
    return a["bus"].noul, a["scene"].choice, a["scene"].confidence, a["people"].score


answers = {}
with S1GHT(f"vllm://127.0.0.1:8000/{REPO}", timeout=300) as remote:
    for name, img in PHOTOS.items():
        t = time.perf_counter()
        answers[("vllm", name)] = (*summary(remote.decide({"photo": img}, QS)), time.perf_counter() - t)
server.terminate()
server.wait(60)
gc.collect()
torch.cuda.empty_cache()

with S1GHT("qwen3-vl-4b", device="cuda") as local:
    for name, img in PHOTOS.items():
        local.decide({"photo": img}, QS)  # warm-up
        t = time.perf_counter()
        answers[("native", name)] = (*summary(local.decide({"photo": img}, QS)), time.perf_counter() - t)
    dtype = local.runtime.dtype

print(torch.cuda.get_device_name(), "native dtype", dtype)
print(f"{'runtime':7s} {'photo':10s} {'bus':>6s} {'scene':18s} {'conf':>5s} {'people':>6s} {'time':>6s}")
for (rt, name), (bus, scene, conf, people, secs) in sorted(answers.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    print(f"{rt:7s} {name:10s} {bus:6.3f} {scene:18s} {conf:5.2f} {people:6.2f} {secs:5.1f}s")

## 4. Latency by image size and batch (`s1ght bench`, issue #22)

Set `BRANCH = "feat/bench"` (or `main` once it is merged) in the first cell, then run the GPU check, clone and install cells, then this one. The section runs in about 15 minutes on a T4 and downloads d1-omni-600M 2.4 GB, d1-3B 6.3 GB and Qwen3-VL-4B 8.9 GB. Paste the Markdown it prints into the PR.

In [ ]:
!cd S1GHT && python benchmarks/latency.py d1-omni-600m:native d1-3b:native qwen3-vl-4b:native --device cuda --sizes 128 256 384 640 --batches 1 4 16 --runs 10 --warmup 2 --max-time 60 --out ../bench_gpu.json

## 5. Qwen3-VL-4B prefix cache (issue #13)

Set `BRANCH = "feat/prefix-cache"` (or `main` once it is merged) in the first cell, then run the GPU check, clone and install cells, then these. No vLLM is needed. Downloads: Qwen3-VL-4B-Instruct 8.9 GB.

The model loads once. Every case runs with the prefix cache off (the old path: every prompt reads the image again) and on (the image is encoded once per request and the questions branch off it). The answers must match up to fp16/bf16 noise; `max |Δp|` is the largest probability difference over all questions. Paste the table into the PR.

In [ ]:
!cd S1GHT && python -m pytest tests/test_prefix_cache.py -m network -q -p no:cacheprovider  # tiny random Qwen3-VL on CPU, ~12 MB download

In [ ]:
import gc
import time

import httpx
import torch

from s1ght import S1GHT, Choice, Image, Noul, Score

ASSETS = "https://github.com/ultralytics/assets/releases/download/v0.0.0"
PHOTOS = {n: Image(httpx.get(f"{ASSETS}/{n}", follow_redirects=True).content) for n in ("bus.jpg", "zidane.jpg")}
QS = {
    "bus": Noul("Is there a bus in `photo`?"),
    "scene": Choice("Where was `photo` taken?", ["city street", "football stadium", "kitchen", "beach"]),
    "people": Score("How many people are in `photo`?", ["none", "one or two", "three or more"]),
    "tie": Noul("Is someone in `photo` wearing a tie?"),
    "light": Choice("What is the lighting in `photo`?", ["daylight", "indoor lights", "night"]),
    "crowd": Score("How crowded is `photo`?", ["empty", "a few people", "busy", "packed"]),
}
SETS = {n: dict(list(QS.items())[:n]) for n in (1, 3, 6)}


def probs(resp):
    out = {}
    for qid, a in resp.answers.items():
        w = a.to_wire()
        out[qid] = [w["noul"]] if w["type"] == "noul" else list(w["probabilities"].values())
    return out


def timed(model, img, qs, n=5, warmup=1):
    for _ in range(warmup):
        resp = model.decide({"photo": img}, qs)
    times = []
    for _ in range(n):
        torch.cuda.synchronize()
        t = time.perf_counter()
        resp = model.decide({"photo": img}, qs)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times), probs(resp)


rows = []
with S1GHT("qwen3-vl-4b", device="cuda") as local:
    family = local.base_runtime.family
    engine = family.prefix  # PrefixCache; None turns the cache off
    for flip in ("auto", "always"):
        family.flip = flip
        for name, img in PHOTOS.items():
            for n, qs in SETS.items():
                family.prefix = None
                plain_ms, plain = timed(local, img, qs)
                family.prefix = engine
                cached_ms, cached = timed(local, img, qs)
                diff = max(abs(x - y) for q in plain for x, y in zip(plain[q], cached[q], strict=True))
                rows.append((flip, name, n, plain_ms, cached_ms, diff))
gc.collect()
torch.cuda.empty_cache()

print(f"GPU: {torch.cuda.get_device_name()}")
print("| flip | photo | questions | cache off ms | cache on ms | speed-up | max abs dp |")
print("| --- | --- | ---: | ---: | ---: | ---: | ---: |")
for flip, name, n, a, b, d in rows:
    print(f"| {flip} | {name} | {n} | {a:.0f} | {b:.0f} | {a / b:.1f}x | {d:.4f} |")

## 6. P2.0: is there location signal in the patch hidden states? (standalone gate)

Set `BRANCH = "feat/patch-signal"` in the first cell, then run the GPU check, clone and install cells, then these. Downloads: Qwen3-VL-4B-Instruct 8.9 GB (skipped if this runtime already has it) and coco128-seg 7 MB (128 COCO images with masks).

For every image and class, Qwen3-VL-4B reads S1GHT's normal Noul prompt once ("Is there a {class} in `image`?"). From that one forward pass, each method turns the patch tokens into a score map, and the map is compared with the COCO masks:

- `cosine`: patch state · decision-token state, without training.
- `attention_*`: the decision token's attention to the patches (YOLOJ's baseline).
- `objectness`: a linear probe, "is this patch inside any object".
- `bilinear`: a small query-conditioned probe; `bilinear_unseen` is graded on classes it never saw.
- `tiles_2` / `tiles_4`: crops each asked the same Noul (black-box baseline).

Half the images train the probes and choose the layers and thresholds; the table reports the other half. My unmeasured estimate is 10 to 30 minutes, depending on the GPU. Paste the printed tables back into the chat.

In [ ]:
!cd S1GHT && python benchmarks/patch_signal.py --device cuda --out ../patch_signal.json

## 7. P2.1: a trained detection head on the frozen decision model (COCO mAP)

Set `BRANCH = "feat/patch-head"` in the first cell, then run the GPU check, clone and install cells, then these.

Section 6 showed that Qwen3-VL-4B knows *whether* an object is there (presence AUC 0.99), but that location does not fall out of its frozen states without training. Here a small head is trained on those states, as detectors do:
- one forward pass per image gives the patch states of layers 4 and 20;
- a 2-layer transformer over the patches gives a score per patch for each class query and a box per patch;
- COCO mAP is measured on val.

20 of the 80 classes are held out from training to measure open-vocabulary detection.

Rows of the table:
- `head`: the head's own scores.
- `head x presence`: the head's scores multiplied by the model's own Noul answer ("is there a {class}?").
- `yolo26n.pt`: a reference on the same pixels. It was trained on all 118k COCO train images, so it is a yardstick, not a like-for-like comparison.

Downloads:
- COCO annotations: 253 MB.
- 4,000 train and 500 val images: about 720 MB.
- Qwen3-VL-4B: 8.9 GB, skipped if this runtime already has it.
- yolo26n.pt: a few MB.

The feature cache needs about 13 GB of disk. Time is not measured yet: the feature pass is one forward per image. On a T4, consider `--train-images 2000`. Paste the printed table back into the chat.

In [ ]:
%pip install -q ultralytics  # only for the yolo26n.pt reference row
!cd S1GHT && python benchmarks/patch_head.py --device cuda --presence --reference yolo26n.pt --out ../patch_head.json